# **Extracción del Contenido Textual del Corpus**

- **TFM:** Evaluación experimental de Recursive Language Models para el análisis automatizado de literatura científica

- **Autor:** Juan Antonio Jiménez Cobo

---



## **Propósito**

Este notebook tiene el objetivo de extraer el contenido textual de los `N_PAPERS` archivos PDF de los registros del corpus alojados en `PDF_DIR`, creando un archivo `.txt` para cada registro y almacenándolo en `RAW_DIR`. Para ello, se llevan a cabo los siguientes pasos:
1. Monta Google Drive, instala las dependencias desde el archivo `requirements.txt` y configura las rutas del repositorio desde el archivo `.env`
2. Asigna un ID a cada registro para emparejarlo con su archivo PDF correspondiente (`paper_01.pdf`, `paper_02.pdf`, ...)
3. Extrae el texto de cada PDF empleando el método `get_text("text")` del módulo `fitz` de la biblioteca `pymupdf`, y estimando el número de tokens de cada archivo con el tokenizador `cl100k_base`
4. Aplica una limpieza básica del texto:
    - Unir palabras separadas por guiones al final de líneas texto
    - Eliminar líneas de texto cortas (<3 caracteres)
    - Eliminar múltiples espacios y saltos de línea
5. Guarda cada artículo como `paper_XX.txt` en la ruta `RAW_DIR`
6. Genera un informe de extracción en CSV con las métricas de extracción de cada registro
7. Muestra estadísticas del corpus extraído, así como los registros con errores durante la extracción
8. Permite inspeccionar el texto extraído de cada registro

---

## 0. Configuración inicial


### 0.1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 0.2. Instalación de dependencias

In [ ]:
!pip install -r /content/drive/MyDrive/TFM/requirements.txt --quiet
print("✓ Dependencias instaladas")

### 0.3. Configuración de rutas

> **IMPORTANTE:** Las rutas se cargan desde el archivo `.env`, ubicado en  el directorio base en Google Drive. En caso de no tener `.env` configurado, se usarán los valores por defecto indicados en el código. Consulta el archivo `.env.example` en el repositorio para ver las variables disponibles

In [ ]:
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env en Google Drive
load_dotenv('/content/drive/MyDrive/proyect/.env')

# Carpeta con los artículos en formato .pdf (paper_01.pdf ... paper_55.pdf)
PDF_DIR = os.getenv('CORPUS_PDF_DIR', '/content/drive/MyDrive/proyect/corpus/papers')

# Ruta al CSV con los metadatos del corpus (exportado mediante Rayyan)
CSV_PATH = os.getenv('CORPUS_METADATA_PATH', '/content/drive/MyDrive/proyect/corpus/metadata/corpus_metadata.csv')

# Carpeta de salida para los archivos .txt con el texto extraído
RAW_DIR = os.getenv('CORPUS_RAW_DIR', '/content/drive/MyDrive/proyect/corpus/raw')

# Carpeta de salida para el informe de extracción
REPORT_DIR = os.getenv('CORPUS_REPORTS_DIR', '/content/drive/MyDrive/proyect/corpus/reports')

# Ruta de salida para el informe de extracción
REPORT_PATH = os.path.join(REPORT_DIR, 'extraction_report.csv')

# Número total de artículos en el corpus
N_PAPERS = 55

## 1. Imports y utilidades

Se configura el tokenizador `cl100k_base` de la biblioteca `tiktoken`, el mismo tokenizador que se emplea en GPT-4, para estimar la longitud en tokens de cada registro procesado.

In [ ]:
import os
import re
import csv
import pandas as pd
import fitz
import tiktoken
from pathlib import Path

# Crear carpetas de salida si no existe
Path(RAW_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)

# Tokenizador cl100k_base
enc = tiktoken.get_encoding("cl100k_base")

# Función para estimar la longitud de una variable de texto en tokens mediante cl100k_base
def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("✓ Imports completados")
print(f"✓ Carpeta de salida de los textos extraídos: {RAW_DIR}")
print(f"✓ Carpeta de salida del informe de extracción: {REPORT_DIR}")

## 2. Cargar metadatos del CSV

Se carga el archivo CSV con los metadatos de cada registro del corpus, y se le añade un identificador (`paper_id`) en función de la posición del registro en el archivo (`paper_01`, ... ,`paper_55`).

> **IMPORTANTE:** Para un correcto funcionamiento del pipeline de extracción, es necesario que el identificador del registro (`paper_XX`) debe coincidir con el nombre del archivo PDF del registro en la ruta `PDF_DIR` (`paper_XX.pdf`).

In [ ]:
df = pd.read_csv(CSV_PATH)

# Añadir columna paper_id
df["paper_id"] = [f"paper_{str(i+1).zfill(2)}" for i in range(len(df))]

print(f"✓ CSV cargado: {len(df)} filas")
print(f"  Columnas: {list(df.columns)}")
df.head(5)

## 3. Funciones de extracción y limpieza

### 3.1. Función de limpieza de texto

Aplica una limpieza básica al texto extraído de los archivos PDF, sin modificar el contenido original. Se emplean las siguientes transformaciones:
- Elimina guiones de separación de línea (`ex-\nample` → `example`)
- Elimina líneas muy cortas, menos de 3 caracteres, que suelen corresponder a cabeceras o pies de página
- Colapsa espacios múltiples y líneas en blanco redundantes


In [ ]:
def clean_text(raw: str) -> str:
    """
    Limpieza básica del texto extraído. No modifica el contenido del texto
    original.
    """
    # 1. Agrupar palabras separadas por guión al final de línea
    text = re.sub(r"-(\n)\s*", "", raw)

    # 2. Eliminar líneas muy cortas (≤ 3 caracteres)
    lines = text.split("\n")
    lines = [l for l in lines if len(l.strip()) > 3]
    text = "\n".join(lines)

    # 3. Colapsar múltiples líneas en blanco a una sola
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 4. Colapsar espacios múltiples
    text = re.sub(r" {2,}", " ", text)

    return text.strip()

### 3.2. Función de extracción de texto

Extrae el texto de un archivo PDF empleando el método `.get_text("text")` del módulo `fitz`, incluido en la biblioteca `pymupdf`. Realiza una limpieza básica con la función `clean_text`, definida previamente. Devuelve tanto el texto extraído como el número de páginas del documento. Devuelve un `ValueError` si el archivo no tiene texto extraíble.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> tuple[str, int]:
    """
    Extrae y limpia el texto de un PDF con pymupdf.
    Devuelve el texto extraido y el número de páginas del archivo.
    Devuelve ValueError si el PDF no tiene texto extraíble.
    """
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    pages_text = []

    for page in doc:
        page_text = page.get_text("text")
        if page_text.strip():
            pages_text.append(page_text)

    doc.close()

    if not pages_text:
        raise ValueError("PDF sin texto extraíble")

    raw_text = "\n".join(pages_text)
    clean = clean_text(raw_text)
    return clean, n_pages

## 4. Pipeline de extracción de texto

Bucle principal que procesa los `N_PAPERS` registros del corpus de forma secuencial. Para cada registro, se construye la ruta al PDF correspondiente y se extrae el texto mediante la función `extract_text_from_pdf` definida anterioremente.

- Si el archivo no existe en la ruta especificada, se marca como `'ERROR'` con su motivo correspondiente
- Si la extracción se realiza exitosamente, el texto limpio se guarda como archivo `.txt` en la ruta `RAW_DIR`, registrando las siguientes métricas acerca de la extracción:
  - `paper_id`: Identificador del registro
  - `title`: Título
  - `status`: Estado de la extracción (`'OK'` o `'ERROR'`)
  - `n_pages`: Número de paginas extraidas
  - `n_tokens`: Número de tokens totales extraídos, estimados mediante `cl100k_base`
  - `n_chars`: Número de caractéres totales extraídos
  - `error`: Error específico, en caso de haber

  Finalmente, se imprime un resumen con el número de registros procesados correctamente y los errores encontrados.

In [ ]:
report = []

# Bucle principal
for idx, row in df.iterrows():

    # Construcción de la ruta del archivo PDF
    paper_id = row["paper_id"]
    pdf_filename = f"{paper_id}.pdf"
    pdf_path = os.path.join(PDF_DIR, pdf_filename)

    title = row.get("title", paper_id) if "title" in df.columns else paper_id
    short_title = str(title)[:60] + "..." if len(str(title)) > 60 else str(title)

    # Métricas de extracción
    record = {
        "paper_id": paper_id,
        "title": title,
        "status": None,
        "n_pages": None,
        "n_tokens": None,
        "n_chars": None,
        "error": None,
    }

    # Error en caso de no encontrar el PDF en la ruta indicada
    if not os.path.exists(pdf_path):
        record["status"] = "ERROR"
        record["error"] = "PDF no encontrado"
        print(f"  ✗ {paper_id} | PDF no encontrado")
        report.append(record)
        continue

    # Si la ruta del archivo PDF es correcta, se realiza la extracción
    try:
        text, n_pages = extract_text_from_pdf(pdf_path)
        n_tokens = count_tokens(text)
        n_chars = len(text)

        # Guardar archivo .txt con el texto extraído
        txt_path = os.path.join(RAW_DIR, f"{paper_id}.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        record["status"] = "OK"
        record["n_pages"] = n_pages
        record["n_tokens"] = n_tokens
        record["n_chars"] = n_chars

        # Impresión de cada registro y sus métricas de extracción
        print(f"  ✓ {paper_id} | {n_pages} págs | {n_tokens:,} tokens | {short_title}")

    # Registro de errores durante la extracción
    except Exception as e:
        record["status"] = "ERROR"
        record["error"] = str(e)
        print(f"  ✗ {paper_id} | ERROR: {e} | {short_title}")

    report.append(record)

# Resumen final del número registros procesados correctamente y número de errores
print("\n" + "="*60)
print(f"Procesados: {len([r for r in report if r['status'] == 'OK'])} / {N_PAPERS}")
print(f"Errores:    {len([r for r in report if r['status'] == 'ERROR'])} / {N_PAPERS}")

## 5. Output final

### 5.1. Informe de extracción

Se guarda un informe de extracción CSV en `REPORT_PATH` con las métricas de extracción de cada registro procesado.

In [ ]:
report_df = pd.DataFrame(report)
report_df.to_csv(REPORT_PATH, index=False, encoding="utf-8")

print(f"✓ Informe guardado en: {REPORT_PATH}")
print()
report_df

### 5.2. Estadísticas del corpus

Para los artículos procesados con éxito (`status = 'OK'`), se imprime una tabla con estadísticas acerca de la extracción del corpus, específicamente:

- Número de artículos procesados
- Total de tokens del corpus
- Media de tokens por artículo
- Mediana de tokens por artículo
- Mínimo de tokens en todos los registros procesados
- Máximo de tokens en todos los registros procesados

Adicionalmente, se imprimen tantos aquellos registros en los que se ha detectado un error durante la extracción, como aquellos en los que la extracción se ha realizado correctamente, pero el número de tokens extraídos no supera un umbral (en este caso 500 tokens), lo cual puede implicar un fallo no detectado durante la extracción.

In [ ]:
ok = report_df[report_df["status"] == "OK"]

if len(ok) > 0:
    print("=" * 50)
    print("ESTADÍSTICAS DEL CORPUS (artículos extraídos correctamente)")
    print("=" * 50)
    print(f"  Artículos procesados:     {len(ok)}")
    print(f"  Total tokens:             {ok['n_tokens'].sum():,}")
    print(f"  Media tokens/artículo:    {ok['n_tokens'].mean():,.0f}")
    print(f"  Mín tokens:               {ok['n_tokens'].min():,} ({ok.loc[ok['n_tokens'].idxmin(), 'paper_id']})")
    print(f"  Máx tokens:               {ok['n_tokens'].max():,} ({ok.loc[ok['n_tokens'].idxmax(), 'paper_id']})")
    print(f"  Mediana tokens/artículo:  {ok['n_tokens'].median():,.0f}")
    print()

    # Artículos con pocos tokens (<500) detectados como sospechosos
    sospechosos = ok[ok["n_tokens"] < 500]
    if len(sospechosos) > 0:
        print("⚠ ARTÍCULOS CON MENOS DE 500 TOKENS (revisar manualmente):")
        for _, r in sospechosos.iterrows():
            print(f"   - {r['paper_id']}: {r['n_tokens']} tokens")
    else:
        print("✓ Ningún artículo sospechoso (todos > 500 tokens)")

# Mostrar artículos con errores
errors = report_df[report_df["status"] == "ERROR"]
if len(errors) > 0:
    print()
    print("✗ ARTÍCULOS CON ERROR:")
    for _, r in errors.iterrows():
        print(f"   - {r['paper_id']}: {r['error']}")

### 5.3. Inspección de registros

Se permite inspeccionar el texto extraído de cada registro para realizar un análisis más detallado sobre la extracción. Cambia `PAPER_TO_INSPECT` con el ID del registro a inspeccionar, y `N_CHARS_PREVIEW` con el número de caracteres a mostrar del registro extraído.

In [ ]:
PAPER_TO_INSPECT = "paper_01"       # registro a mostrar
N_CHARS_PREVIEW = 2000              # número de caracteres a mostrar

txt_path = os.path.join(RAW_DIR, f"{PAPER_TO_INSPECT}.txt")

if os.path.exists(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"--- Primeros {N_CHARS_PREVIEW} caracteres de {PAPER_TO_INSPECT} ---\n")
    print(content[:N_CHARS_PREVIEW])
    print(f"\n[...] ({len(content):,} caracteres totales)")
else:
    print(f"Archivo no encontrado: {txt_path}")